In [1]:
!pip -q install -U transformers accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 97.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.8 MB/s eta 0:00:00


In [2]:
from huggingface_hub import notebook_login

notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
import os
import re
import json
import time
from datetime import datetime, UTC

import numpy as np
import pandas as pd
from sentence_transformers import CrossEncoder

# REQUIRES MANUAL UPLOAD TO RUNTIME FROM LOCAl
JSONL_PATH = "/content/prep_rows.jsonl"
CROSS_ENCODER_MODEL = "cross-encoder/ms-marco-MiniLM-L6-v2"

# TRY WITH 10 TO TEST
# REPLACE WITH len(prep_rows) FOR FULL RUN
N_RUN = 10 
TARGET_REL = 2
SAVE_EVERY = 1
N_PRINT = 5

RUN_ROOT = "/content/trecdl_cross_encoder_runs"
os.makedirs(RUN_ROOT, exist_ok=True)

RUN_NAME = "cross_encoder_full100_" + datetime.now(UTC).strftime("%Y%m%d_%H%M%S")
RUN_DIR = os.path.join(RUN_ROOT, RUN_NAME)
os.makedirs(RUN_DIR, exist_ok=True)

def write_json(path, obj):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

def norm_ws(s):
    return re.sub(r"\s+", " ", (s or "")).strip()

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows

def precision_at_k(labels, ranked_indices, k, rel_threshold):
    top = ranked_indices[:k]
    if len(top) < k:
        return None
    good = sum(1 for ix in top if labels[ix] >= rel_threshold)
    return good / k

def recall_at_k(labels, ranked_indices, k, rel_threshold):
    top = ranked_indices[:k]
    if len(top) < k:
        return None
    total_good = sum(1 for x in labels if x >= rel_threshold)
    if total_good == 0:
        return 0.0
    good = sum(1 for ix in top if labels[ix] >= rel_threshold)
    return good / total_good

def hit_at_k(labels, ranked_indices, k, rel_threshold):
    top = ranked_indices[:k]
    if len(top) < k:
        return None
    return float(any(labels[ix] >= rel_threshold for ix in top))

def dcg_at_k(labels, ranked_indices, k):
    top = ranked_indices[:k]
    if len(top) < k:
        return None
    vals = []
    for rank_pos, ix in enumerate(top, start=1):
        rel = labels[ix]
        gain = (2 ** max(rel, 0) - 1) / np.log2(rank_pos + 1)
        vals.append(gain)
    return float(sum(vals))

def ndcg_at_k(labels, ranked_indices, k):
    actual = dcg_at_k(labels, ranked_indices, k)
    if actual is None:
        return None
    ideal_order = sorted(range(len(labels)), key=lambda i: labels[i], reverse=True)
    ideal = dcg_at_k(labels, ideal_order, k)
    if ideal == 0:
        return 0.0
    return actual / ideal

def cross_encoder_rank(query, candidates, ce_model):
    pairs = [(query, c["passage_text"]) for c in candidates]
    scores = ce_model.predict(pairs, show_progress_bar=False)
    scores = np.asarray(scores, dtype=float)
    order = sorted(range(len(scores)), key=lambda x: scores[x], reverse=True)
    return order, scores.tolist()

if not os.path.exists(JSONL_PATH):
    raise FileNotFoundError(f"Uploaded JSONL file not found: {JSONL_PATH}")

prep_rows = load_jsonl(JSONL_PATH)

for ex in prep_rows:
    ex["trec_year"] = int(ex["trec_year"])
    ex["query_id"] = str(ex["query_id"])
    ex["query"] = norm_ws(ex["query"])
    ex["labels_100"] = [int(x) for x in ex["labels_100"]]
    ex["highly_rel_positions"] = [int(x) for x in ex["highly_rel_positions"]]
    ex["n_candidates"] = int(ex["n_candidates"])
    ex["n_highly_relevant"] = int(ex["n_highly_relevant"])

    clean_candidates = []
    for c in ex["candidates_100"]:
        clean_candidates.append({
            "doc_id": str(c["doc_id"]),
            "score": float(c["score"]),
            "passage_text": norm_ws(c["passage_text"]),
            "source_doc_id": str(c.get("source_doc_id", ""))
        })
    ex["candidates_100"] = clean_candidates

prep_rows = sorted(prep_rows, key=lambda x: (x["trec_year"], x["query_id"]))

print("Loaded prep_rows:", len(prep_rows))
print()

year_counts = (
    pd.DataFrame([{"trec_year": x["trec_year"]} for x in prep_rows])
    .groupby("trec_year")
    .size()
    .reset_index(name="n_queries")
)

print("Prepared queries by year")
print(year_counts.to_string(index=False))
print()

shape_df = pd.DataFrame([
    {
        "trec_year": x["trec_year"],
        "query_id": x["query_id"],
        "n_candidates": x["n_candidates"],
        "n_highly_relevant": x["n_highly_relevant"]
    }
    for x in prep_rows
])

print("Prepared dataset shape summary")
print(shape_df[["n_candidates", "n_highly_relevant"]].describe().to_string())
print()

bad_rows = []
for i, ex in enumerate(prep_rows):
    if len(ex["candidates_100"]) != 100:
        bad_rows.append((i, ex["trec_year"], ex["query_id"], "candidates_100_len", len(ex["candidates_100"])))
    if len(ex["labels_100"]) != 100:
        bad_rows.append((i, ex["trec_year"], ex["query_id"], "labels_100_len", len(ex["labels_100"])))
    if ex["n_candidates"] != 100:
        bad_rows.append((i, ex["trec_year"], ex["query_id"], "n_candidates_field", ex["n_candidates"]))
    if ex["n_highly_relevant"] < 5:
        bad_rows.append((i, ex["trec_year"], ex["query_id"], "n_highly_relevant_field", ex["n_highly_relevant"]))

print("Validation check")
if bad_rows:
    print("Found invalid rows:", len(bad_rows))
    for row in bad_rows[:20]:
        print(row)
    raise ValueError("Dataset validation failed")
else:
    print("All rows passed basic validation")
print()

for ex in prep_rows[:N_PRINT]:
    print("=" * 120)
    print("TREC YEAR:", ex["trec_year"])
    print("QUERY ID:", ex["query_id"])
    print("QUERY:", ex["query"])
    print("N CANDIDATES:", ex["n_candidates"])
    print("N HIGHLY RELEVANT:", ex["n_highly_relevant"])
    print("TOP HIGHLY RELEVANT POSITIONS:", ex["highly_rel_positions"][:10])
    print()

    preview_rows = []
    for j, c in enumerate(ex["candidates_100"][:5]):
        preview_rows.append({
            "cand_ix": j,
            "doc_id": c["doc_id"],
            "score": round(c["score"], 4),
            "relevance": ex["labels_100"][j],
            "passage_preview": c["passage_text"][:220]
        })

    print(pd.DataFrame(preview_rows).to_string(index=False))
    print()

print("Loading cross-encoder model")
print("Model name:", CROSS_ENCODER_MODEL)
print()

ce_model = CrossEncoder(CROSS_ENCODER_MODEL)

run_rows = prep_rows[:N_RUN]

meta = {
    "run_name": RUN_NAME,
    "created_utc": datetime.now(UTC).strftime("%Y-%m-%d %H:%M:%S"),
    "jsonl_path": JSONL_PATH,
    "n_loaded_queries": int(len(prep_rows)),
    "n_run": int(len(run_rows)),
    "target_rel": int(TARGET_REL),
    "model_name": CROSS_ENCODER_MODEL
}
write_json(os.path.join(RUN_DIR, "meta.json"), meta)

print("Run configuration")
print(json.dumps(meta, indent=2))
print()

results_rows = []
raw_rows = []

for idx, ex in enumerate(run_rows):
    t0 = time.time()

    query = ex["query"]
    labels = ex["labels_100"]
    candidates = ex["candidates_100"]

    ranking, scores = cross_encoder_rank(query, candidates, ce_model)

    elapsed = time.time() - t0

    top3 = ranking[:3]
    top5 = ranking[:5]

    p3 = precision_at_k(labels, ranking, 3, TARGET_REL)
    r3 = recall_at_k(labels, ranking, 3, TARGET_REL)
    hit3 = hit_at_k(labels, ranking, 3, TARGET_REL)
    ndcg3 = ndcg_at_k(labels, ranking, 3)

    p5 = precision_at_k(labels, ranking, 5, TARGET_REL)
    r5 = recall_at_k(labels, ranking, 5, TARGET_REL)
    hit5 = hit_at_k(labels, ranking, 5, TARGET_REL)
    ndcg5 = ndcg_at_k(labels, ranking, 5)

    result_row = {
        "idx": int(idx),
        "trec_year": int(ex["trec_year"]),
        "query_id": ex["query_id"],
        "elapsed_sec": float(elapsed),
        "n_candidates": int(len(candidates)),
        "n_highly_relevant": int(sum(1 for x in labels if x >= TARGET_REL)),
        "top3": top3,
        "top5": top5,
        "p_at_3_rel2": p3,
        "r_at_3_rel2": r3,
        "hit_at_3_rel2": hit3,
        "ndcg_at_3": ndcg3,
        "p_at_5_rel2": p5,
        "r_at_5_rel2": r5,
        "hit_at_5_rel2": hit5,
        "ndcg_at_5": ndcg5
    }
    results_rows.append(result_row)

    raw_row = {
        "idx": int(idx),
        "trec_year": int(ex["trec_year"]),
        "query_id": ex["query_id"],
        "query": ex["query"],
        "scores": scores
    }
    raw_rows.append(raw_row)

    if (idx + 1) % SAVE_EVERY == 0:
        write_jsonl(os.path.join(RUN_DIR, "results_rows.jsonl"), results_rows)
        write_jsonl(os.path.join(RUN_DIR, "raw_rows.jsonl"), raw_rows)

    done = idx + 1
    valid_ndcg3 = [r["ndcg_at_3"] for r in results_rows if r["ndcg_at_3"] is not None]
    valid_ndcg5 = [r["ndcg_at_5"] for r in results_rows if r["ndcg_at_5"] is not None]

    mean_ndcg3 = float(np.mean(valid_ndcg3)) if valid_ndcg3 else None
    mean_ndcg5 = float(np.mean(valid_ndcg5)) if valid_ndcg5 else None

    print(
        f"{done}/{len(run_rows)} done | "
        f"mean_ndcg3={mean_ndcg3 if mean_ndcg3 is not None else 'NA'} | "
        f"mean_ndcg5={mean_ndcg5 if mean_ndcg5 is not None else 'NA'} | "
        f"last_time={elapsed:.2f}s"
    )

write_jsonl(os.path.join(RUN_DIR, "results_rows.jsonl"), results_rows)
write_jsonl(os.path.join(RUN_DIR, "raw_rows.jsonl"), raw_rows)

results_df = pd.DataFrame(results_rows)

summary = {
    "n_queries": int(len(results_df)),
    "mean_elapsed_sec": float(results_df["elapsed_sec"].mean()) if len(results_df) else None,
    "metrics": {
        "p_at_3_rel2": float(results_df["p_at_3_rel2"].dropna().mean()) if len(results_df["p_at_3_rel2"].dropna()) else None,
        "r_at_3_rel2": float(results_df["r_at_3_rel2"].dropna().mean()) if len(results_df["r_at_3_rel2"].dropna()) else None,
        "hit_at_3_rel2": float(results_df["hit_at_3_rel2"].dropna().mean()) if len(results_df["hit_at_3_rel2"].dropna()) else None,
        "ndcg_at_3": float(results_df["ndcg_at_3"].dropna().mean()) if len(results_df["ndcg_at_3"].dropna()) else None,
        "p_at_5_rel2": float(results_df["p_at_5_rel2"].dropna().mean()) if len(results_df["p_at_5_rel2"].dropna()) else None,
        "r_at_5_rel2": float(results_df["r_at_5_rel2"].dropna().mean()) if len(results_df["r_at_5_rel2"].dropna()) else None,
        "hit_at_5_rel2": float(results_df["hit_at_5_rel2"].dropna().mean()) if len(results_df["hit_at_5_rel2"].dropna()) else None,
        "ndcg_at_5": float(results_df["ndcg_at_5"].dropna().mean()) if len(results_df["ndcg_at_5"].dropna()) else None
    }
}

write_json(os.path.join(RUN_DIR, "summary.json"), summary)

print()
print("Run finished")
print("Run directory:", RUN_DIR)
display(results_df)
print(json.dumps(summary, indent=2))

Loaded prep_rows: 125

Prepared queries by year
 trec_year  n_queries
      2021         40
      2022         42
      2023         43

Prepared dataset shape summary
       n_candidates  n_highly_relevant
count         125.0         125.000000
mean          100.0          14.936000
std             0.0          10.383566
min           100.0           5.000000
25%           100.0           8.000000
50%           100.0          10.000000
75%           100.0          20.000000
max           100.0          52.000000

Validation check
All rows passed basic validation

TREC YEAR: 2021
QUERY ID: 1040198
QUERY: who is the final arbiter of florida law in instances where there is no federal authority?
N CANDIDATES: 100
N HIGHLY RELEVANT: 8
TOP HIGHLY RELEVANT POSITIONS: [0, 1, 2, 3, 7, 9, 16, 62]

 cand_ix                       doc_id   score  relevance                                                                                                                                                

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Run configuration
{
  "run_name": "cross_encoder_full100_20260412_235826",
  "created_utc": "2026-04-12 23:58:28",
  "jsonl_path": "/content/prep_rows.jsonl",
  "n_loaded_queries": 125,
  "n_run": 125,
  "target_rel": 2,
  "model_name": "cross-encoder/ms-marco-MiniLM-L6-v2"
}

1/125 done | mean_ndcg3=0.7318407279869963 | mean_ndcg5=0.7002532264119935 | last_time=0.05s
2/125 done | mean_ndcg3=0.8659203639934981 | mean_ndcg5=0.8501266132059968 | last_time=0.05s
3/125 done | mean_ndcg3=0.8260187442911325 | mean_ndcg5=0.7902560260751356 | last_time=0.06s
4/125 done | mean_ndcg3=0.7913009372145566 | mean_ndcg5=0.7861653186774156 | last_time=0.06s
5/125 done | mean_ndcg3=0.6983319661256495 | mean_ndcg5=0.7199436311646392 | last_time=0.05s
6/125 done | mean_ndcg3=0.6980650885354467 | mean_ndcg5=0.6943107780790431 | last_time=0.05s
7/125 done | mean_ndcg3=0.6787202279700872 | mean_ndcg5=0.6701912875483069 | last_time=0.05s
8/125 done | mean_ndcg3=0.7188801994738263 | mean_ndcg5=0.7114173766047

,idx,trec_year,query_id,elapsed_sec,n_candidates,n_highly_relevant,top3,top5,p_at_3_rel2,r_at_3_rel2,hit_at_3_rel2,ndcg_at_3,p_at_5_rel2,r_at_5_rel2,hit_at_5_rel2,ndcg_at_5
0,0,2021,1040198,0.053918,100,8,"[0, 1, 2]","[0, 1, 2, 9, 7]",1.000000,0.375000,1.0,0.731841,1.0,0.625000,1.0,0.700253
1,1,2021,1104300,0.054309,100,52,"[2, 3, 4]","[2, 3, 4, 7, 40]",1.000000,0.057692,1.0,1.000000,1.0,0.096154,1.0,1.000000
2,2,2021,1104447,0.059314,100,39,"[10, 0, 9]","[10, 0, 9, 1, 5]",0.666667,0.051282,1.0,0.746216,0.6,0.076923,1.0,0.670515
3,3,2021,1107821,0.055294,100,12,"[6, 11, 1]","[6, 11, 1, 2, 28]",0.666667,0.166667,1.0,0.687148,0.8,0.333333,1.0,0.773893
4,4,2021,1109840,0.053789,100,13,"[0, 1, 2]","[0, 1, 2, 6, 69]",0.333333,0.076923,1.0,0.326456,0.6,0.230769,1.0,0.455057
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
120,120,2023,3100289,0.055008,100,6,"[0, 1, 3]","[0, 1, 3, 11, 25]",0.666667,0.333333,1.0,0.765361,0.6,0.500000,1.0,0.684352
121,121,2023,3100569,0.046939,100,5,"[36, 18, 3]","[36, 18, 3, 25, 49]",0.333333,0.200000,1.0,0.458058,0.2,0.200000,1.0,0.438154
122,122,2023,3100825,0.046110,100,6,"[13, 22, 34]","[13, 22, 34, 8, 14]",0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.043735
123,123,2023,3100909,0.043666,100,9,"[14, 15, 9]","[14, 15, 9, 44, 54]",0.666667,0.222222,1.0,0.798881,0.6,0.333333,1.0,0.742184


{
  "n_queries": 125,
  "mean_elapsed_sec": 0.05186237144470215,
  "metrics": {
    "p_at_3_rel2": 0.6906666666666667,
    "r_at_3_rel2": 0.18603842209095098,
    "hit_at_3_rel2": 0.896,
    "ndcg_at_3": 0.663156093740496,
    "p_at_5_rel2": 0.6208,
    "r_at_5_rel2": 0.2720261240907832,
    "hit_at_5_rel2": 0.944,
    "ndcg_at_5": 0.6374474810346554
  }
}
